# # 13. State Representation Learning (SRL): Masked Transformer

En este notebook implementamos la tercera y última arquitectura de nuestra fase de extracción de características: el **Masked Transformer**. Este modelo utiliza el mecanismo de **Self-Attention** para comprender las dependencias globales dentro de una ventana de 24 horas de Bitcoin.

A diferencia de las RNNs (LSTM/GRU), el Transformer procesa toda la secuencia en paralelo, permitiendo capturar relaciones entre eventos lejanos en el tiempo que una red recurrente podría olvidar. Al "enmascarar" parte de los datos, obligamos al modelo a aprender la estructura interna y la correlación entre las diferentes escalas de tiempo (Wavelets) para reconstruir la información faltante.

## 1. Estrategia de Enmascaramiento 

Para este entrenamiento, aplicaremos un enmascaramiento del 20% sobre la secuencia de entrada. 
- **Entrada**: Una ventana de 24 velas.
- **Proceso**: Seleccionamos 5 velas al azar y sustituimos sus valores por 0.
- **Tarea**: El Transformer debe utilizar la información de las 19 velas restantes para inferir los valores de las 5 ocultas.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.append('../')
from src.utils import create_sliding_windows,prepare_multivariate_data
from src.srl_models import MaskedTransformerSRL
from torch.utils.data import DataLoader, TensorDataset

# --- AJUSTES PARA SEÑAL RUIDOSA (Sin Wavelets) ---
WINDOW_SIZE = 168
BATCH_SIZE = 64

LEARNING_RATE = 0.0005  # Bajamos un orden de magnitud
EPOCHS = 150            # Necesitamos más tiempo para converger
EMBED_DIM = 128          # Aumentamos la capacidad de representación
NHEAD = 8               # Mantenemos cabezas de atención
NUM_LAYERS = 4          # Un poco más de profundidad
MASK_PROB = 0.15        # Bajamos un poco la máscara (menos agresiva)
BLOCK_SIZE = 4

from_safe = '2021-12-31_00-00-00'
until_safe = '2025-07-31_00-00-00'

foldername = "SP500"
#foldername = "BTC"
data_name = "SPY"
#data_name = "BTCUSDT"

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo detectado: {device}")
if torch.cuda.is_available():
    print(f"Ferrari detectado: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ ATENCIÓN: Sigues en el bus (CPU).")

Dispositivo detectado: cuda
Ferrari detectado: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Arquitectura Detallada
A diferencia de las arquitecturas recurrentes, el Transformer utiliza el mecanismo de Scaled Dot-Product Attention. Esto permite que el modelo calcule la relevancia de cada paso de tiempo en relación con todos los demás dentro de la misma ventana de 24 horas.La importancia de un dato $x_i$ frente a un dato $x_j$ se define mediante la fórmula:$$Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$Donde:

**Queries (Q)**: Representan lo que buscamos en el estado actual.

**Keys (K)**: Representan la información disponible de otros momentos.

**Values (V)**: Es el contenido que finalmente extraemos una vez determinada la relevancia.

In [3]:
from sklearn.preprocessing import RobustScaler
def apply_block_mask(batch, mask_prob=0.15, block_size=4):
    """Borra bloques de tiempo para obligar al modelo a entender tendencias."""
    batch_size, seq_len, feat_dim = batch.shape
    masked_batch = batch.clone()
    mask = torch.zeros(batch_size, seq_len, dtype=torch.bool)
    
    for i in range(batch_size):
        num_masked = int(seq_len * mask_prob)
        num_blocks = max(1, num_masked // block_size)
        for _ in range(num_blocks):
            start = torch.randint(0, seq_len - block_size, (1,)).item()
            mask[i, start:start+block_size] = True
    
    masked_batch[mask] = 0.0
    return masked_batch, mask

## 3. Proceso de enmascaramiento
El aprendizaje en este notebook es de tipo Auto-supervisado. La "tarea de pretexto" consiste en reconstruir valores faltantes. Este enfoque es superior al aprendizaje tradicional por dos motivos técnicos:

**Entendimiento Estructural**: Para "adivinar" una vela de las 14:00 que ha sido borrada, la IA debe entender la correlación con la vela de las 08:00 y las 20:00. No puede simplemente "copiar" el valor anterior.

**Regularización Intrínseca**: El enmascaramiento actúa como un dropout dinámico y agresivo, impidiendo que el modelo se aprenda de memoria el ruido del Bitcoin y forzándolo a aprender patrones de volatilidad y volumen generales.

In [4]:
#["1h", "1d"]
for tf in ["1h"]:
    print(f"\n--- Entrenando Masked Transformer para {tf.upper()} ---")
    
    file_path = f'../data/{foldername}/01-output-{data_name}_{tf}-from-{from_safe}-until-{until_safe}-log-return.csv'
    df = pd.read_csv(file_path, parse_dates=['date'], index_col='date')
    
    ### NUEVO: Preparar las 3 variables ###
    df = prepare_multivariate_data(df)
    feature_cols = [
        'normalized_outliers_processed_log_return', 
        'normalized_range', 
        'normalized_tradecount'
    ]
    features = df[feature_cols]
    
    print(f"Configuración: Entrenando con {features.shape[1]} variables -> {feature_cols}")
    
    # 2. Preparación de Tensores
    X_tensor = create_sliding_windows(features, WINDOW_SIZE)
    dataset = TensorDataset(X_tensor)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    # 3. Inicializar Modelo (input_dim ahora es 3)
    input_dim = features.shape[1]
    model = MaskedTransformerSRL(input_dim, EMBED_DIM, NHEAD, NUM_LAYERS).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=10, factor=0.5)
    criterion = nn.MSELoss()

    # 4. Bucle de Entrenamiento
    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch in dataloader:
            optimizer.zero_grad()
            inputs = batch[0].to(device)
            
            ### NUEVO: Máscara por BLOQUES ###
            masked_inputs, mask = apply_block_mask(inputs, MASK_PROB, BLOCK_SIZE)
            
            reconstructed, _ = model(masked_inputs)
            
            # El error se calcula solo en la zona borrada de las 3 variables
            loss = criterion(reconstructed[mask], inputs[mask])
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss/len(dataloader)
        scheduler.step(avg_loss)
        
        if (epoch+1) % 1 == 0:
            print(f"Época {epoch+1}/{EPOCHS}, Masked MSE: {avg_loss:.6f}")

    # 5. Guardar Modelo
    torch.save(model.state_dict(), f'../results/{foldername}/transformer_srl_{tf}.pth')

    


--- Entrenando Masked Transformer para 1H ---
Configuración: Entrenando con 3 variables -> ['normalized_outliers_processed_log_return', 'normalized_range', 'normalized_tradecount']
Época 1/150, Masked MSE: 0.927323
Época 2/150, Masked MSE: 0.927979
Época 3/150, Masked MSE: 0.953201
Época 4/150, Masked MSE: 0.867737
Época 5/150, Masked MSE: 0.844333
Época 6/150, Masked MSE: 0.822037
Época 7/150, Masked MSE: 0.854906
Época 8/150, Masked MSE: 0.908861
Época 9/150, Masked MSE: 0.819282
Época 10/150, Masked MSE: 0.861886
Época 11/150, Masked MSE: 0.813272
Época 12/150, Masked MSE: 0.816167
Época 13/150, Masked MSE: 0.876157
Época 14/150, Masked MSE: 0.869042
Época 15/150, Masked MSE: 0.940066
Época 16/150, Masked MSE: 0.918551
Época 17/150, Masked MSE: 0.878801
Época 18/150, Masked MSE: 0.812575
Época 19/150, Masked MSE: 0.792078
Época 20/150, Masked MSE: 0.875398
Época 21/150, Masked MSE: 0.858106
Época 22/150, Masked MSE: 0.849868
Época 23/150, Masked MSE: 0.817633
Época 24/150, Masked M

In [5]:
# --- GENERACIÓN DE EMBEDDINGS TRANSFORMER (CORREGIDO) ---
# No hace falta entrenar de nuevo, el modelo ya está en 'model' y guardado en disco

model.eval()
all_embeddings = []

# Usamos el eval_loader que ya definiste (shuffle=False es clave)
eval_loader = DataLoader(dataset, batch_size=BATCH_SIZE * 2, shuffle=False)

print(f"Generando embeddings para {len(dataset)} ventanas en la GPU...")

with torch.no_grad():
    for batch in eval_loader:
        # EL TRUCO: Mover el batch al mismo sitio que el modelo (GPU)
        inputs = batch[0].to(device)
        
        # Obtenemos el embedding latente
        _, latent = model(inputs)
        
        # IMPORTANTE: Mover el resultado a la CPU antes de guardarlo
        all_embeddings.append(latent.cpu())

# Unificamos todos los tensores (ahora todos están en CPU)
full_latent = torch.cat(all_embeddings, dim=0).numpy()

# Alineamos con el índice temporal (saltando las WINDOW_SIZE filas iniciales)
adjusted_index = df.index[len(df) - len(full_latent):]

# Creamos el DataFrame
df_embeddings = pd.DataFrame(full_latent, index=adjusted_index)
df_embeddings.columns = [f'trans_embedding_{i}' for i in range(EMBED_DIM)]

# Guardamos el CSV final
output_path = f'../data/{foldername}/02-srl-transformer-{tf}-from-{from_safe}-until-{until_safe}.csv'
df_embeddings.to_csv(output_path)

print(f"✅ ÉXITO TOTAL: Embeddings del Transformer guardados en: {output_path}")
print(f"Dimensiones: {df_embeddings.shape}")

Generando embeddings para 1971 ventanas en la GPU...
✅ ÉXITO TOTAL: Embeddings del Transformer guardados en: ../data/SP500/02-srl-transformer-1h-from-2021-12-31_00-00-00-until-2025-07-31_00-00-00.csv
Dimensiones: (1971, 128)


## 4. Análisis de métrica
El error se calcula únicamente sobre los datos ocultos, por lo que cualquier reducción de este significa que el Transformer es capaz de desarrollar una intuición sobre la estructura del Bitcoin.

En el marco de **1h** vemos una reducción del error del 60% (de 0.048 a 0.019).
La importancia de esto es que el modelo no está solo memorizando, sino que está utilizando la atención global para inferir valores de volatilidad y volumen basandose en el contexto previo.

En el marco de **1d**, el error se estabiliza en torno al 0.070, es un resultado esperado ya que el mercado diario tiene un ratio señal-ruido mucho menor que el horario. Que no logre bajar el error demuestra que la arquitectura es robusta frente al overfitting.



## 5. Visualización t-SNE: El Espacio de Atención
Proyectamos los embeddings de 32 dimensiones generados por el Transformer para observar cómo el mecanismo de atención agrupa los estados del mercado.



In [6]:
from sklearn.manifold import TSNE

def plot_transformer_results(tf_to_plot="1h"):
    # 1. Cargar embeddings generados
    path = f'../data/02-srl-transformer-{tf_to_plot}-from-{from_safe}-until-{until_safe}.csv'
    df_embeddings = pd.read_csv(path, index_col='date')
    
    # Cargar retornos originales para colorear
    df_orig = pd.read_csv(f'../data/01-output-BTCUSDT_{tf_to_plot}-from-{from_safe}-until-{until_safe}-log-return.csv', 
                          parse_dates=['date'], index_col='date')
    
    # 2. Reducción de dimensionalidad
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    sample_size = 2000 if tf_to_plot == "1h" else len(df_embeddings)
    data_2d = tsne.fit_transform(df_embeddings.iloc[:sample_size])
    
    # 3. Plot
    plt.figure(figsize=(12, 6))
    returns = df_orig['processed_log_return'].loc[df_embeddings.index[:sample_size]]
    
    sc = plt.scatter(data_2d[:, 0], data_2d[:, 1], c=returns, cmap='RdYlGn', alpha=0.5, s=10)
    plt.colorbar(sc, label='Log-Return (Dirección)')
    plt.title(f'Mapa Latente Transformer - Atención Global ({tf_to_plot.upper()})')
    plt.grid(alpha=0.2)
    plt.show()

#plot_transformer_results("1h")

## 6. Análisis de los resultados
Tras comparar los tres modelos de SRL, observamos que:

El **Autoencoder** es excelente capturando la forma exacta de la señal (baja pérdida de reconstrucción).

El **CPC** es óptimo capturando la dirección futura (alta precisión de contraste).

El **Transformer** genera el espacio latente con mayor separación de regímenes (clusters definidos), lo que sugiere que es el modelo que mejor filtra el ruido aleatorio del mercado mediante sus cabezales de atención.